# A Practical Guide to Quantitative Finance Interviews — Chapter 2 Brain Teasers

**Xinfeng Zhou ("the Green Book")** — worked solutions in code.

This notebook grows as I read through the book. Each brain teaser gets:

* a short **restatement** of the problem,
* a **markdown explanation of the logic**, and
* **one general-purpose function** that solves *every* instance (all $n$, arbitrary
  parameters) — not just the specific numbers in the book.

Covered so far:

| § | Theme | Problems |
|---|-------|----------|
| **2.1** | Problem Simplification | Screwy pirates · Tiger and sheep |
| **2.2** | Logic Reasoning | River crossing · Birthday problem · Card game · Burning ropes · Defective ball · Trailing zeros · Horse race · Infinite power tower |

The recurring meta-technique of §2.1 is: *when a problem is too big, solve the
smallest case, then grow it.* In code that is just **dynamic programming /
recursion** — build the answer for size $n$ from the answer for size $n-1$.

---
## 2.1 Problem Simplification

> *If the original problem is too complex to solve at once, identify a simplified
> version and start there — the simplest sub-problem — and gradually add
> complexity until a pattern emerges.*

### 2.1.1 Screwy pirates

**Problem.** $N$ pirates (ranked by seniority) split $M$ gold coins. The most
senior proposes a split; **all** present pirates vote; if **at least half**
approve, it passes, otherwise the proposer is thrown overboard and the next
senior proposes. Every pirate is perfectly rational with priorities:
**(1) survive, (2) maximise gold, (3) all else equal, prefer fewer pirates left.**
How are the coins divided?

**Logic.** The 5-pirate case is hopeless head-on, so *simplify*:

* **1 pirate:** takes all $M$ coins.
* **2 pirates:** the proposer's own vote is already 50% → he keeps everything,
  junior gets 0.
* **3 pirates:** if the proposer dies we fall back to the 2-pirate outcome where
  pirate 1 gets 0. So pirate 1 accepts **1 coin** (strictly better than 0). Two
  votes → passes.
* **$p$ pirates:** the proposer needs $\lceil p/2\rceil$ votes (his own plus
  $\lceil p/2\rceil-1$ bribes). A junior sells his vote for **one more coin than
  he'd get if the proposer died** — i.e. than his payoff in the already-solved
  $(p-1)$-pirate game. So bribe the *cheapest* juniors (those who'd get 0, or who'd
  *die* in the sub-game and vote yes just to live).

That is a **bottom-up DP**: `dp[p]` is built from `dp[p-1]`. It also captures the
famous coin-limited regime — with only 100 coins a very senior proposer may be
unable to buy enough votes and gets thrown overboard.

In [1]:
from math import ceil

def screwy_pirates(n_pirates, coins):
    """Rational pirate split. Pirates 1..N by seniority (N = first proposer).
    Returns payoff[1..N] (index 0 unused); a pirate thrown overboard is 'dead'.
    Bottom-up DP: the p-pirate outcome is built from the (p-1)-pirate sub-game."""
    dp = {1: {1: coins}}                      # lone pirate takes everything
    for p in range(2, n_pirates + 1):
        prev = dp[p - 1]                      # outcome if proposer p is thrown over
        bribes_needed = ceil(p / 2) - 1       # 'at least 50%' minus his own vote
        # price of each junior's vote: dead-in-sub-game -> 0 (wants to live),
        # else one coin more than his sub-game payoff
        cost = {j: (0 if prev.get(j, 'dead') == 'dead' else prev[j] + 1)
                for j in range(1, p)}
        cheapest = sorted(range(1, p), key=cost.get)[:bribes_needed]
        bill = sum(cost[j] for j in cheapest)
        if bill <= coins:                     # proposer can buy a majority
            dp[p] = {p: coins - bill,
                     **{j: (cost[j] if j in cheapest else 0) for j in range(1, p)}}
        else:                                 # can't -> thrown overboard
            dp[p] = {**prev, p: 'dead'}
    final = dp[n_pirates]
    return [None] + [final.get(i, 'dead') for i in range(1, n_pirates + 1)]

# Book case: 5 pirates, 100 coins  ->  [1, 0, 1, 0, 98]
print("5 pirates, 100 coins:", screwy_pirates(5, 100)[1:])
for n in (2, 3, 4, 5, 6, 10):
    print(f"{n:>2} pirates, 100 coins:", screwy_pirates(n, 100)[1:])
# Coin-limited regime: proposer #203 cannot buy enough votes and dies
r = screwy_pirates(205, 100)
print("205 pirates: proposer #205 ->", r[205], "| survivors =",
      sum(x != 'dead' for x in r[1:]))

5 pirates, 100 coins: [1, 0, 1, 0, 98]
 2 pirates, 100 coins: [0, 100]
 3 pirates, 100 coins: [1, 0, 99]
 4 pirates, 100 coins: [0, 1, 0, 99]
 5 pirates, 100 coins: [1, 0, 1, 0, 98]
 6 pirates, 100 coins: [0, 1, 0, 1, 0, 98]
10 pirates, 100 coins: [0, 1, 0, 1, 0, 1, 0, 1, 0, 96]
205 pirates: proposer #205 -> dead | survivors = 204


### 2.1.2 Tiger and sheep

**Problem.** $n$ perfectly-rational tigers and 1 sheep are on an island. A tiger
that eats the sheep *becomes* a sheep (and is then edible by the others). Tigers
prefer to eat, but survival comes first. Is the sheep eaten?

**Logic.** Simplify by induction on $n$:

* $n=1$: the lone tiger eats — nothing can punish it.
* $n=2$: eating turns you into a sheep facing 1 hungry tiger → you'd be eaten. So
  neither eats. Sheep **safe**.
* In general a tiger eats **iff** the resulting $(n-1)$-tiger world is *safe* for
  its new sheep-self. Hence `eaten(n) = not eaten(n-1)`, with `eaten(1)=True`.

The pattern: the sheep is eaten **iff $n$ is odd**.

In [2]:
from functools import lru_cache

@lru_cache(None)
def sheep_eaten(n_tigers):
    """True iff a lone sheep is eaten by n rational tigers.
    eaten(n) = not eaten(n-1); eaten(1) = True  ->  odd n means eaten."""
    if n_tigers <= 0:
        return False
    return not sheep_eaten(n_tigers - 1)

for n in [1, 2, 3, 4, 5, 99, 100]:
    print(f"{n:>3} tigers -> sheep {'EATEN' if sheep_eaten(n) else 'safe'}")

  1 tigers -> sheep EATEN
  2 tigers -> sheep safe
  3 tigers -> sheep EATEN
  4 tigers -> sheep safe
  5 tigers -> sheep EATEN
 99 tigers -> sheep EATEN
100 tigers -> sheep safe


---
## 2.2 Logic Reasoning

> *Carefully chain what each fact/observation implies, discarding impossibilities
> until only the answer remains.*

### 2.2.1 River crossing (bridge & torch)

**Problem.** People must cross a bridge that holds **2 at a time**; a single torch
must travel on every crossing, so a pair moves at the **slower** person's pace.
Minimise total time. (Book: times $10,5,2,1 \Rightarrow 17$ min.)

**Logic.** Sort times ascending. The bottleneck is the two **slowest**; the trick
is to ferry them across *together* so their times overlap, using the fastest
people as shuttles. Getting the two slowest ($y \le z$) across, with the two
fastest ($a \le b$) available, costs the cheaper of two schedules:

* **A — send the fast pair as ferries:** $\;2b + a + z$
  ($a,b$ over; $a$ back; $y,z$ over; $b$ back).
* **B — fastest shuttles each slow one:** $\;2a + y + z$.

Peel off the two slowest, add the cheaper cost, repeat until $\le 3$ remain (whose
optima are immediate). This greedy-DP is optimal for **any** number of people and
**any** times.

In [3]:
def bridge_and_torch(times):
    """Minimum time to move everyone across a 2-person bridge with one torch.
    General for any number of people and any crossing times.
    Returns (total_time, moves)."""
    t = sorted(times); n = len(t)
    if n <= 2:
        return (t[-1] if t else 0,
                [("cross", tuple(t))] if t else [])
    moves, total, hi = [], 0, n - 1
    while hi >= 3:
        a, b, y, z = t[0], t[1], t[hi - 1], t[hi]      # 2 fastest, 2 slowest
        if 2 * b + a + z <= 2 * a + y + z:             # A: fast pair ferries
            moves += [("cross", (a, b)), ("back", (a,)),
                      ("cross", (y, z)), ("back", (b,))]
            total += 2 * b + a + z
        else:                                          # B: fastest shuttles
            moves += [("cross", (a, z)), ("back", (a,)),
                      ("cross", (a, y)), ("back", (a,))]
            total += 2 * a + y + z
        hi -= 2
    if hi == 2:                                        # last three
        moves += [("cross", (t[0], t[1])), ("back", (t[0],)), ("cross", (t[0], t[2]))]
        total += t[1] + t[0] + t[2]
    else:                                              # last two
        moves += [("cross", (t[0], t[1]))]; total += t[1]
    return total, moves

total, moves = bridge_and_torch([10, 5, 2, 1])
print("Book case [10,5,2,1] -> total =", total, "min")
for m in moves:
    print("  ", m)
print("General [1,2,5,10,15,20] ->", bridge_and_torch([1, 2, 5, 10, 15, 20])[0], "min")

Book case [10,5,2,1] -> total = 17 min
   ('cross', (1, 2))
   ('back', (1,))
   ('cross', (5, 10))
   ('back', (2,))
   ('cross', (1, 2))
General [1,2,5,10,15,20] -> 42 min


### 2.2.2 Birthday problem

**Problem.** Boss $A$'s birthday is one of 10 dates. **You** are told only the
**month**, colleague **C** only the **day**. Then:

1. *You:* "I don't know $A$'s birthday, **and I know C doesn't either**."
2. *C:* "Now I know it."
3. *You:* "Now I know it too."

What is the birthday? (Green Book dates: Mar 4/5/8, Jun 4/7, Sep 1/5, Dec 1/2/8.)

**Logic** — each statement is a *filter* on the candidate set:

1. You are certain **C** can't know ⇒ your month contains **no globally-unique
   day** ⇒ eliminate every month that holds a day appearing only once (that kills
   the months of `Jun 7` and `Dec 2`, i.e. June and December).
2. **C** now knows ⇒ among survivors his **day is unique** ⇒ keep dates whose day
   is unique (kills the day shared by March and September).
3. **You** now know ⇒ among survivors your **month is unique**.

One date survives. This engine works for **any** date set + this statement
pattern.

In [4]:
from collections import Counter

def birthday_deduction(dates):
    """Solve the 'boss's birthday' common-knowledge puzzle for any date set.
    dates: list of (month, day). Applies the three public statements as filters."""
    uniq = lambda pool, idx: {v for v, k in Counter(p[idx] for p in pool).items() if k == 1}
    S = list(dates)
    bad_months = {m for (m, d) in S if d in uniq(S, 1)}    # (1) you're sure C can't know
    S = [x for x in S if x[0] not in bad_months]
    S = [x for x in S if x[1] in uniq(S, 1)]               # (2) C now knows -> day unique
    S = [x for x in S if x[0] in uniq(S, 0)]               # (3) you now know -> month unique
    return S

green_book = [("Mar", 4), ("Mar", 5), ("Mar", 8), ("Jun", 4), ("Jun", 7),
              ("Sep", 1), ("Sep", 5), ("Dec", 1), ("Dec", 2), ("Dec", 8)]
print("A's birthday is:", birthday_deduction(green_book))

A's birthday is: [('Sep', 1)]


### 2.2.3 Card game

**Problem.** A deck has $R$ red and $B$ black cards (book: $26$ each). Cards are
turned two at a time: **red-red** → your pile, **black-black** → dealer's pile,
**mixed** → discarded. You win \$100 if your pile is **strictly larger**. What is
the game worth?

**Logic (invariant / symmetry).** Every *mixed* discard removes exactly one red
and one black. So each red card ends up either in your pile or in a mixed discard,
and likewise each black. Counting:

$$\text{(your pile)} - \text{(dealer pile)} = R - B \quad\text{— always, for every shuffle.}$$

With $R=B$ the two piles are **always equal** → you can never have *more* → the
game is worth **\$0**. (More generally you're guaranteed to win iff $R>B$.)

In [5]:
def card_game_value(reds, blacks, payoff=100):
    """Value of the red/black pairing game. Invariant: your_pile - dealer_pile
    == reds - blacks for every arrangement, so the outcome is guaranteed."""
    diff = reds - blacks
    if diff > 0:
        return payoff, "You always win."
    if diff < 0:
        return 0, "You can never win -> pay nothing."
    return 0, "Always a tie -> you never have MORE -> pay nothing."

print("26 red / 26 black:", card_game_value(26, 26))
print("28 red / 24 black:", card_game_value(28, 24))

26 red / 26 black: (0, 'Always a tie -> you never have MORE -> pay nothing.')
28 red / 24 black: (100, 'You always win.')


### 2.2.4 Burning ropes

**Problem.** Two ropes each burn for **60 min** but *unevenly* (you can't trust
any fraction of a rope to take a proportional time). Measure **45 min**.

**Logic.** Lighting a rope at **both ends** always finishes in $T/2$ *regardless*
of the uneven density (the two flames jointly consume the whole rope). So:

* Light rope A at **both** ends and rope B at **one** end.
* When A is gone (**30 min**), B has exactly 30 min of burn left — now light B's
  **other** end, halving it → **15 min** more. Total **45 min**.

Generalisation: repeatedly halving lets you measure any $60\cdot(\text{dyadic
rational})$; the function lists the reachable durations for $k$ ropes.

In [6]:
from fractions import Fraction

def burning_ropes_45():
    """Constructive schedule to measure 45 min with two 60-min ropes."""
    return ["t=0 : light rope A at BOTH ends, rope B at ONE end",
            "t=30: A burned out -> light B's OTHER end (B has 30 min left -> halves to 15)",
            "t=45: B burned out. Total = 45 minutes"]

def rope_measurable_times(n_ropes, unit=60, halvings=3):
    """Durations reachable with n identical `unit`-minute ropes by lighting ends:
    unit * (dyadic rationals) up to n*unit."""
    vals = {Fraction(w) + Fraction(k, 2 ** m)
            for w in range(n_ropes + 1) for m in range(halvings + 1)
            for k in range(2 ** m + 1)}
    return sorted(v * unit for v in vals if 0 < v <= n_ropes)

for line in burning_ropes_45():
    print(line)
print("Reachable with 2 ropes (min):", [round(float(v), 2) for v in rope_measurable_times(2)])

t=0 : light rope A at BOTH ends, rope B at ONE end
t=30: A burned out -> light B's OTHER end (B has 30 min left -> halves to 15)
t=45: B burned out. Total = 45 minutes
Reachable with 2 ropes (min): [7.5, 15.0, 22.5, 30.0, 37.5, 45.0, 52.5, 60.0, 67.5, 75.0, 82.5, 90.0, 97.5, 105.0, 112.5, 120.0]


### 2.2.5 Defective ball

**Problem.** Among $n$ identical balls exactly one is defective — **heavier *or*
lighter**, you don't know which. Using a balance (tells which pan is heavier, or
balance), find the defective ball **and** whether it's heavy/light. (Book:
$n=12$ in 3 weighings.)

**Logic (information counting → ternary codes).** Each weighing has 3 outcomes
(left down / balance / right down), so $k$ weighings distinguish $3^k$ cases. There
are $2n$ possibilities ($n$ balls × heavy/light); we also must reject the "no
weighing moved" contradiction and keep heavy/light distinguishable, giving the
classic bound

$$\frac{3^{k}-3}{2} \ge n .$$

**Construction (one plan for all $n$).** Give each ball a distinct code in
$\{L,R,N\}^k$: in weighing $c$ put its $L$-balls left, $R$-balls right. Choose codes
so that (i) no code and its $L\leftrightarrow R$ **mirror** are both used (so a
heavy ball and its mirror light ball never collide) and (ii) every column has
equal $L$s and $R$s (pans balance). The observed outcome string then **is** the
defective ball's code (heavy) or its mirror (light). $n=12,k=3$ falls straight out.

In [7]:
def min_weighings(n, direction_known=False):
    """Fewest balance weighings for one defective among n balls.
    unknown heavy/light: (3^k-3)/2 >= n;  known direction: 3^k >= n."""
    k = 0
    while (3 ** k if direction_known else (3 ** k - 3) // 2) < n:
        k += 1
    return k

_mirror = lambda code: tuple('R' if c == 'L' else 'L' if c == 'R' else 'N' for c in code)

def build_weighing_plan(n, k=None):
    """General non-adaptive plan: assign each of n balls a {L,R,N}^k code with no
    used mirror pair and balanced pans. Returns (codes, weighings)."""
    from itertools import product
    if k is None:
        k = min_weighings(n)
    pairs, seen = [], set()
    for c in product('LRN', repeat=k):                 # canonical rep per mirror pair
        if all(x == 'N' for x in c) or c in seen:
            continue
        seen.add(c); seen.add(_mirror(c))
        pairs.append(c if next(x for x in c if x != 'N') == 'L' else _mirror(c))
    vec = lambda code: [1 if x == 'L' else -1 if x == 'R' else 0 for x in code]
    chosen = []
    def bt(i, run, need):                              # pick n pairs, sign them -> pans balance
        if need == 0:
            return all(v == 0 for v in run)
        if i >= len(pairs) or len(pairs) - i < need:
            return False
        v = vec(pairs[i])
        for s in (1, -1):
            chosen.append(pairs[i] if s == 1 else _mirror(pairs[i]))
            if bt(i + 1, [run[c] + s * v[c] for c in range(k)], need - 1):
                return True
            chosen.pop()
        return bt(i + 1, run, need)                    # or skip this pair
    if not bt(0, [0] * k, n):
        raise ValueError(f"n={n} needs more than k={k} weighings")
    codes = list(chosen)
    weighings = [([b for b in range(n) if codes[b][c] == 'L'],
                  [b for b in range(n) if codes[b][c] == 'R']) for c in range(k)]
    return codes, weighings

def find_defective(n, defective, heavier, plan=None):
    """Identify the defective ball and its type from the plan's outcomes."""
    codes, weighings = plan or build_weighing_plan(n)
    obs = []
    for left, right in weighings:
        d = 1 if heavier else -1
        w = (d if defective in left else 0) - (d if defective in right else 0)
        obs.append('L' if w > 0 else 'R' if w < 0 else 'N')
    obs = tuple(obs)
    for b, code in enumerate(codes):
        if obs == code:            return b, True     # its own side sank -> heavy
        if obs == _mirror(code):   return b, False    # its own side rose  -> light
    raise RuntimeError("undecodable")

print("min weighings: n=12 ->", min_weighings(12), "| n=100 ->", min_weighings(100))
plan = build_weighing_plan(12)
print("Weighings for 12 balls (left pan vs right pan):")
for i, (L, R) in enumerate(plan[1], 1):
    print(f"  W{i}: {L} vs {R}")
# verify every one of the 24 hidden scenarios is solved correctly
ok = all(find_defective(12, b, h, plan) == (b, h) for b in range(12) for h in (True, False))
print("All 24 hidden (ball, heavy/light) scenarios identified in 3 weighings:", ok)

min weighings: n=12 -> 3 | n=100 -> 5
Weighings for 12 balls (left pan vs right pan):
  W1: [0, 1, 2, 3] vs [4, 5, 6, 7]
  W2: [0, 1, 2, 4] vs [3, 8, 9, 10]
  W3: [0, 3, 6, 9] vs [1, 5, 8, 11]
All 24 hidden (ball, heavy/light) scenarios identified in 3 weighings: True


### 2.2.6 Trailing zeros

**Problem.** How many trailing zeros does $100!$ have?

**Logic.** A trailing zero is a factor of $10 = 2\times5$. In $n!$ factors of $2$
vastly outnumber factors of $5$, so **count the 5s** (Legendre's formula):

$$Z_5(n) = \left\lfloor \tfrac n5\right\rfloor + \left\lfloor \tfrac n{25}\right\rfloor + \left\lfloor \tfrac n{125}\right\rfloor + \cdots$$

For $n=100$: $20 + 4 = 24$. Generalised to any base $b$: factor $b=\prod p^{e}$,
count each prime's exponent in $n!$, divide by $e$, take the minimum.

In [8]:
def _factorize(m):
    f, d = {}, 2
    while d * d <= m:
        while m % d == 0:
            f[d] = f.get(d, 0) + 1; m //= d
        d += 1
    if m > 1:
        f[m] = f.get(m, 0) + 1
    return f

def trailing_zeros_factorial(n, base=10):
    """Trailing zeros of n! in the given base (Legendre's formula)."""
    zeros = None
    for p, e in _factorize(base).items():
        cnt, pk = 0, p
        while pk <= n:
            cnt += n // pk; pk *= p
        zeros = cnt // e if zeros is None else min(zeros, cnt // e)
    return zeros or 0

print("trailing zeros of 100! =", trailing_zeros_factorial(100))
print("trailing zeros of 1000! =", trailing_zeros_factorial(1000))
print("trailing zeros of 100! in base 12 =", trailing_zeros_factorial(100, 12))

trailing zeros of 100! = 24
trailing zeros of 1000! = 249
trailing zeros of 100! in base 12 = 48


### 2.2.7 Horse race

**Problem.** $25$ horses, a track with $5$ lanes, no timer. Fewest races to find
the **3 fastest**? (Answer: **7**.)

**Logic.**

1. **5 group races** (horses 1-5, …, 21-25) rank each group of 5.
2. **1 race of the 5 group winners** → the overall **fastest**, and it prunes whole
   groups: only a group whose winner placed 1st/2nd/3rd can contribute more
   top-3 horses.
3. After that, exactly **5 horses** remain eligible for places 2 and 3 (the
   triangular pruning leaves $\binom{m+1}{2}-1$ contenders) → **1 final race**.

Total $5+1+1 = 7$. The function returns the count for general $n$, lanes, and
top-$m$ (exact for the usual $m \le \text{lanes}$).

In [9]:
import math

def horse_races(n_horses, lanes, top_m):
    """Races (standard method) to find the fastest top_m of n horses, `lanes`
    at a time, no clock. Returns (races, explanation)."""
    if top_m > n_horses:
        raise ValueError("top_m > n_horses")
    groups = math.ceil(n_horses / lanes)
    races = groups + 1                                   # group races + winners' race
    if top_m == 1:
        return races, "winner of the winners' race is fastest"
    contenders = top_m * (top_m + 1) // 2 - 1            # eligible for places 2..top_m
    extra = math.ceil(contenders / lanes)
    return races + extra, f"{groups} group + 1 winners' + {extra} runoff"

print("25 horses, 5 lanes, top 3 ->", horse_races(25, 5, 3))
print("49 horses, 7 lanes, top 3 ->", horse_races(49, 7, 3))

25 horses, 5 lanes, top 3 -> (7, "5 group + 1 winners' + 1 runoff")
49 horses, 7 lanes, top 3 -> (9, "7 group + 1 winners' + 1 runoff")


### 2.2.8 Infinite sequence (power tower)

**Problem.** Solve $x^{x^{x^{\cdot^{\cdot^{\cdot}}}}} = 2$.

**Logic.** Let $y$ be the whole (infinite) tower. Because it is infinite, the
exponent *is itself* $y$, so $x^{y}=y$. With $y=2$: $x^{2}=2 \Rightarrow x=\sqrt2$.

General: $x^{y}=y \Rightarrow x = y^{1/y}$. **Caveat:** the tower only *converges*
for $x\in[e^{-e},\,e^{1/e}]$, i.e. targets $y\in(0,e]$. So $y=2$ gives
$x=\sqrt2$ (valid), while "solving" $y=4$ also gives $x=\sqrt2$ — but that tower
actually converges to $2$, not $4$ (the classic trap). The function returns $x$
**and** a convergence check.

In [10]:
import math

def tower_base_for(target):
    """Solve x^(x^(x^...)) = target -> x = target**(1/target).
    Returns (x, converges, actual_limit). Tower converges only for target in (0, e]."""
    x = target ** (1.0 / target)
    converges = math.exp(-math.e) <= x <= math.exp(1.0 / math.e)
    v = 1.0
    for _ in range(1000):
        v = x ** v
    return x, converges, v

for target in (2.0, 4.0):
    x, conv, lim = tower_base_for(target)
    note = "OK" if abs(lim - target) < 1e-6 else f"TRAP: tower actually -> {lim:.2f}"
    print(f"target={target}: x = target**(1/target) = {x:.6f}  ({note})")

target=2.0: x = target**(1/target) = 1.414214  (OK)
target=4.0: x = target**(1/target) = 1.414214  (TRAP: tower actually -> 2.00)


---
*Next up as I keep reading: §2.3 Thinking Out of the Box, §2.4 Application of
Symmetry, …*